# Notebook 01 — Data Cleaning
**Proyek**: Analisis Demografi Provinsi Indonesia  
**Sumber**: BPS Indonesia (bps.go.id)  
**Deskripsi**: Membersihkan dan menstandardisasi dataset demografi gabungan provinsi Indonesia sebelum analisis.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Data

In [2]:
RAW_PATH = '../data/raw/demografi_provinsi_indonesia_merged.csv'
META_PATH = '../data/raw/metadata_demografi_provinsi_indonesia.csv'

df_raw = pd.read_csv(RAW_PATH)
df_meta = pd.read_csv(META_PATH)

print(f'Shape raw: {df_raw.shape}')
print(f'\nKolom: {df_raw.columns.tolist()}')
df_raw.head(5)

Shape raw: (45, 26)

Kolom: ['id', 'provinsi', 'jumlah_penduduk_ribu_2026', 'laju_pertumbuhan_pct_2026', 'distribusi_pct_2026', 'kepadatan_per_km2_2026', 'rasio_jenis_kelamin_2026', 'kepadatan_per_km2_2021', 'laju_1971_1980', 'laju_1980_1990', 'laju_1990_2000', 'laju_2000_2010', 'laju_2010_2016', 'laju_2010_2017', 'laju_2010_2018', 'laju_2010_2019', 'laju_2010_2020', 'laju_2020_2021', 'laju_2020_2022', 'laju_2020_2023', 'laju_2020_2024', 'u5mr_per1000_2020', 'cbr_per1000_2020', 'tfr_2020', 'imr_per1000_2020', 'catatan']


,id,provinsi,jumlah_penduduk_ribu_2026,laju_pertumbuhan_pct_2026,distribusi_pct_2026,kepadatan_per_km2_2026,rasio_jenis_kelamin_2026,kepadatan_per_km2_2021,laju_1971_1980,laju_1980_1990,laju_1990_2000,laju_2000_2010,laju_2010_2016,laju_2010_2017,laju_2010_2018,laju_2010_2019,laju_2010_2020,laju_2020_2021,laju_2020_2022,laju_2020_2023,laju_2020_2024,u5mr_per1000_2020,cbr_per1000_2020,tfr_2020,imr_per1000_2020,catatan
0,1,ACEH,5695.90,1.34,1.98,100.00,100.90,92,2.93,2.72,1.46,"2,36 1",2.01,1.98,1.96,1.93,1.56,1.49,1.43,1.41,1.39,22.88,19.64,2.42,19.41,NaN
1,2,BALI,4488.20,0.68,1.56,804.00,100.20,755,1.69,1.18,1.31,2.15,1.21,1.20,1.18,1.17,1.01,1.40,1.29,0.73,0.71,15.37,15.42,2.04,13.26,NaN
2,3,BANTEN,12641.30,1.05,4.40,1351.00,103.10,1248,-,-,3.21,2.78,2.23,2.20,2.17,2.14,1.10,1.76,1.66,1.22,1.16,16.15,16.51,2.01,13.83,NaN
3,4,BENGKULU,2163.30,1.28,0.75,108.00,104.10,102,4.39,4.38,2.97,1.67,1.69,1.67,1.65,1.63,1.55,1.48,1.40,1.35,1.32,23.38,NaN,NaN,19.73,NaN
4,5,DI YOGYAKARTA,3802.70,0.63,1.32,1199.00,97.70,1185,1.10,0.57,0.72,1.04,1.18,1.17,1.16,1.15,0.58,1.61,1.44,0.67,0.65,12.62,13.69,1.89,10.90,NaN


## 2. Inspeksi Awal

In [3]:
print('=== INFO TIPE DATA ===')
print(df_raw.dtypes)
print()
print('=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])
print()
print(f'=== DUPLIKAT ===')
dupes = df_raw[df_raw['provinsi'].duplicated(keep=False)]
print(f'Baris duplikat: {len(dupes)}')
print(dupes[['id','provinsi','catatan']].sort_values('provinsi').to_string())

=== INFO TIPE DATA ===
id                             int64
provinsi                      object
jumlah_penduduk_ribu_2026    float64
laju_pertumbuhan_pct_2026    float64
distribusi_pct_2026          float64
kepadatan_per_km2_2026       float64
rasio_jenis_kelamin_2026     float64
kepadatan_per_km2_2021        object
laju_1971_1980                object
laju_1980_1990                object
laju_1990_2000                object
laju_2000_2010                object
laju_2010_2016                object
laju_2010_2017                object
laju_2010_2018                object
laju_2010_2019                object
laju_2010_2020                object
laju_2020_2021                object
laju_2020_2022                object
laju_2020_2023               float64
laju_2020_2024               float64
u5mr_per1000_2020            float64
cbr_per1000_2020             float64
tfr_2020                     float64
imr_per1000_2020             float64
catatan                       object
dtype: object



**Temuan**:
- Terdapat 8 baris duplikat untuk Gorontalo (kemungkinan dari penggabungan file yang berbeda)
- Beberapa kolom laju historis bertipe `object` karena ada nilai non-numerik (`'-'`, `'2,36 1'`)
- Missing values terbanyak pada indikator SP2020 untuk provinsi pemekar baru

## 3. Deduplikasi

In [4]:
# Keputusan: ambil baris pertama untuk setiap provinsi
# Alasan: baris pertama memiliki data paling lengkap dan catatan NaN (= data bersih)
df = df_raw.drop_duplicates(subset='provinsi', keep='first').copy()
df = df.reset_index(drop=True)

print(f'Sebelum deduplikasi : {len(df_raw)} baris')
print(f'Setelah deduplikasi  : {len(df)} baris (38 provinsi)')

Sebelum deduplikasi : 45 baris
Setelah deduplikasi  : 38 baris (38 provinsi)


## 4. Cleaning Kolom Laju Historis (string → float)

In [5]:
laju_str_cols = [
    'laju_1971_1980', 'laju_1980_1990', 'laju_1990_2000', 'laju_2000_2010',
    'laju_2010_2016', 'laju_2010_2017', 'laju_2010_2018',
    'laju_2010_2019', 'laju_2010_2020', 'laju_2020_2021', 'laju_2020_2022'
]

def clean_laju(val):
    """Membersihkan nilai laju: hapus suffix teks, ganti koma desimal, set '-' jadi NaN."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ['-', '', 'NA', 'N/A']:
        return np.nan
    # Hapus karakter non-numerik di akhir (misal: '2,36 1' → '2.36')
    s = s.replace(',', '.').split()[0]
    try:
        return float(s)
    except ValueError:
        return np.nan

for col in laju_str_cols:
    df[col] = df[col].apply(clean_laju)

# kepadatan_per_km2_2021 juga string
df['kepadatan_per_km2_2021'] = df['kepadatan_per_km2_2021'].apply(clean_laju)

print('Cek tipe data setelah cleaning:')
print(df[laju_str_cols].dtypes)

Cek tipe data setelah cleaning:
laju_1971_1980    float64
laju_1980_1990    float64
laju_1990_2000    float64
laju_2000_2010    float64
laju_2010_2016    float64
laju_2010_2017    float64
laju_2010_2018    float64
laju_2010_2019    float64
laju_2010_2020    float64
laju_2020_2021    float64
laju_2020_2022    float64
dtype: object


## 5. Standardisasi Nama Provinsi

In [6]:
df['provinsi'] = df['provinsi'].str.strip().str.upper()

# Tambah label pulau untuk segmentasi analisis
JAWA = {'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'DI YOGYAKARTA', 'BANTEN'}
SUMATERA = {'ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'KEPULAUAN RIAU',
            'JAMBI', 'BENGKULU', 'SUMATERA SELATAN', 'KEPULAUAN BANGKA BELITUNG', 'LAMPUNG'}
KALIMANTAN = {'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN',
              'KALIMANTAN TIMUR', 'KALIMANTAN UTARA'}
SULAWESI = {'SULAWESI UTARA', 'GORONTALO', 'SULAWESI TENGAH', 'SULAWESI BARAT',
            'SULAWESI SELATAN', 'SULAWESI TENGGARA'}
PAPUA_NTB = {'PAPUA', 'PAPUA BARAT', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN',
             'PAPUA BARAT DAYA', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR'}

def assign_pulau(prov):
    if prov in JAWA: return 'Jawa'
    if prov in SUMATERA: return 'Sumatera'
    if prov in KALIMANTAN: return 'Kalimantan'
    if prov in SULAWESI: return 'Sulawesi'
    if prov in PAPUA_NTB: return 'Papua/NTT/NTB'
    return 'Lainnya'

df['pulau'] = df['provinsi'].apply(assign_pulau)
print(df.groupby('pulau').size().sort_values(ascending=False))

pulau
Sumatera         10
Papua/NTT/NTB     8
Sulawesi          6
Jawa              6
Kalimantan        5
Lainnya           3
dtype: int64


## 6. Validasi Final

In [7]:
print(f'Jumlah provinsi unik : {df["provinsi"].nunique()}')
print(f'Total penduduk 2026  : {df["jumlah_penduduk_ribu_2026"].sum()/1000:.1f} juta jiwa')
print(f'Distribusi total     : {df["distribusi_pct_2026"].sum():.1f}% (harus ~100)')
print()
print('Missing values per kolom utama:')
key_cols = ['jumlah_penduduk_ribu_2026','laju_pertumbuhan_pct_2026','kepadatan_per_km2_2026',
            'laju_2020_2024','imr_per1000_2020','cbr_per1000_2020','tfr_2020']
print(df[key_cols].isnull().sum())

Jumlah provinsi unik : 38
Total penduduk 2026  : 287.2 juta jiwa
Distribusi total     : 100.0% (harus ~100)

Missing values per kolom utama:
jumlah_penduduk_ribu_2026    0
laju_pertumbuhan_pct_2026    0
kepadatan_per_km2_2026       0
laju_2020_2024               0
imr_per1000_2020             5
cbr_per1000_2020             6
tfr_2020                     6
dtype: int64


## 7. Simpan Data Processed

In [8]:
OUT_PATH = '../data/processed/demografi_clean.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

Saved: ../data/processed/demografi_clean.csv
Shape: (38, 27)


,id,provinsi,jumlah_penduduk_ribu_2026,laju_pertumbuhan_pct_2026,distribusi_pct_2026,kepadatan_per_km2_2026,rasio_jenis_kelamin_2026,kepadatan_per_km2_2021,laju_1971_1980,laju_1980_1990,laju_1990_2000,laju_2000_2010,laju_2010_2016,laju_2010_2017,laju_2010_2018,laju_2010_2019,laju_2010_2020,laju_2020_2021,laju_2020_2022,laju_2020_2023,laju_2020_2024,u5mr_per1000_2020,cbr_per1000_2020,tfr_2020,imr_per1000_2020,catatan,pulau
0,1,ACEH,5695.90,1.34,1.98,100.00,100.90,92.00,2.93,2.72,1.46,2.36,2.01,1.98,1.96,1.93,1.56,1.49,1.43,1.41,1.39,22.88,19.64,2.42,19.41,NaN,Sumatera
1,2,BALI,4488.20,0.68,1.56,804.00,100.20,755.00,1.69,1.18,1.31,2.15,1.21,1.20,1.18,1.17,1.01,1.40,1.29,0.73,0.71,15.37,15.42,2.04,13.26,NaN,Lainnya
2,3,BANTEN,12641.30,1.05,4.40,1351.00,103.10,1248.00,NaN,NaN,3.21,2.78,2.23,2.20,2.17,2.14,1.10,1.76,1.66,1.22,1.16,16.15,16.51,2.01,13.83,NaN,Jawa
